## Preprocesamiento de imágenes y construcción del dataset

### Objetivo

Antes de entrenar los modelos de aprendizaje automático y aprendizaje profundo, fue necesario realizar un proceso de preprocesamiento que permitiera transformar las imágenes originales en un conjunto de datos estructurado, limpio y adecuado para las tareas de clasificación de calidad y tamaño.

Este proceso tuvo cuatro objetivos principales:

1. Detectar y aislar cada fruta presente en las imágenes.
2. Extraer características geométricas y cromáticas relevantes.
3. Generar etiquetas de tamaño de manera automática.
4. Reducir el desbalance de clases mediante técnicas de aumento de datos.


### 1. Organización inicial de los datos

Las imágenes originales fueron organizadas según dos criterios:

* Tipo de fruta.
* Categoría de calidad (good, regular y bad).

A partir de esta estructura se construyó un repositorio centralizado que permitió procesar cada categoría de forma independiente y almacenar posteriormente los recortes generados y la información asociada a cada muestra.

### 2. Detección y extracción de frutas

Las imágenes originales podían contener una o varias frutas dentro del mismo encuadre. Debido a esto, fue necesario identificar automáticamente la ubicación de cada fruta y generar recortes individuales que sirvieran como entrada para los modelos de clasificación. Se implementó una estrategia híbrida de detección basada en varios métodos complementarios.

En primer lugar, se utilizó el modelo YOLOv8 para detectar automáticamente frutas cuyas clases estaban disponibles dentro del modelo preentrenado, como manzanas, naranjas y bananos.

Cuando YOLO no encontraba objetos válidos, se aplicaban métodos alternativos basados en visión por computador clásica:

* Segmentación utilizando el canal de saturación en el espacio HSV.
* Detección de contornos mediante agrupación de bordes.
* Segmentación utilizando umbralización y análisis de regiones.

Finalmente, si ninguno de los métodos lograba identificar una fruta, se utilizaba la imagen completa como región candidata.

Este enfoque permitió aumentar la robustez del sistema y evitar la pérdida de muestras durante el procesamiento.


### 3. Eliminación de detecciones duplicadas

Las detecciones obtenidas por los diferentes métodos podían superponerse entre sí.

Para evitar generar múltiples recortes de una misma fruta, se aplicó el algoritmo **Non-Maximum Suppression (NMS)**, el cual conserva únicamente las regiones más representativas y elimina aquellas con un alto grado de solapamiento.

### 4. Filtrado de recortes no válidos

No todos los recortes generados contenían información útil para el entrenamiento. Algunas regiones correspondían a fondos, sombras o detecciones incorrectas.

Por esta razón se implementó un proceso de validación automática. Un recorte era descartado cuando:

* Su tamaño era demasiado pequeño.
* Presentaba muy poca variabilidad de color.
* Mostraba una saturación prácticamente uniforme.
* Contenía una cantidad insuficiente de bordes o textura visual.

Estos criterios permitieron conservar únicamente imágenes con información visual relevante para la clasificación.

### 5. Normalización de las imágenes

Una vez validados los recortes, todas las imágenes fueron redimensionadas a una resolución fija de:

$$224 \times 224 \text{ píxeles}$$

Esta dimensión fue seleccionada porque constituye un tamaño estándar para modelos de visión por computador y permite mantener un equilibrio adecuado entre detalle visual y costo computacional.

### 6. Extracción de características geométricas

Para cada fruta se calcularon diferentes descriptores geométricos que posteriormente fueron utilizados por los modelos clásicos (SVM y XGBoost).

Entre las características extraídas se encuentran:

* Ancho del objeto.
* Alto del objeto.
* Área en píxeles.
* Relación de aspecto.
* Área relativa respecto a otras frutas de la imagen.
* Porcentaje de cobertura de la imagen.

Estas variables describen la forma y escala de cada fruta y resultan especialmente útiles para la tarea de clasificación por tamaño.

### 7. Extracción de características de color

Además de la información geométrica, se extrajeron estadísticas del espacio de color HSV.

Se calcularon:

* Media del matiz (Hue).
* Desviación estándar del matiz.
* Media de la saturación.
* Desviación estándar de la saturación.
* Media del brillo (Value).
* Desviación estándar del brillo.

La elección del espacio HSV se debe a que separa la información de color de la iluminación, permitiendo una representación más robusta frente a cambios de brillo y condiciones de captura. Estas características fueron utilizadas principalmente para la clasificación de calidad.


### 8. Generación automática de etiquetas de tamaño

El conjunto de datos original no incluía etiquetas explícitas de tamaño. Por esta razón se generaron automáticamente utilizando medidas geométricas derivadas de cada fruta. Primero se calculó el área normalizada:

$$
\text{Área Normalizada} =
\frac{\text{Área de la fruta}}{\text{Área total de la imagen}}
$$

Posteriormente, para cada tipo de fruta se calcularon los percentiles 33 y 66 de esta distribución.

Las etiquetas fueron asignadas de la siguiente manera:

* **Pequeño:** Menor al percentil 33
* **Mediano:** Entre los percentiles 33 y 66
* **Grande:** Mayor al percentil 66

Este procedimiento permitió adaptar los umbrales a las características propias de cada fruta.

### 9. Análisis del desbalance de clases

Antes del entrenamiento se evaluó la distribución de muestras por categoría de calidad. Se utilizó el **Imbalance Ratio (IR)** como medida de desbalance:

$$
IR=
\frac{\text{Clase Mayoritaria}}
{\text{Clase Minoritaria}}
$$

Cuando el valor de IR superaba 3, la categoría era considerada críticamente desbalanceada.


### 10. Aumento de datos (data augmentation)

El objetivo es reducir el desbalance entre clases y mejorar la capacidad de generalización de los modelos. Las muestras pertenecientes a clases minoritarias fueron aumentadas mediante:

* Rotaciones aleatorias.
* Volteos horizontales.
* Volteos verticales.
* Variaciones de brillo y contraste.
* Adición de ruido gaussiano.
* Pequeños desplazamientos, escalados y rotaciones.

Estas transformaciones permiten generar nuevas muestras sintéticas conservando la información semántica de la fruta original.


### 11. Construcción del dataset final

Finalmente, toda la información obtenida fue consolidada en un archivo de metadatos que contiene:

* Etiqueta de calidad.
* Etiqueta de tamaño.
* Tipo de fruta.
* Características geométricas.
* Características cromáticas.
* Información de ubicación y dimensiones del recorte.

Este conjunto de datos sirvió como base para el entrenamiento de los modelos SVM y XGBoost, mientras que las imágenes procesadas fueron utilizadas directamente por las redes neuronales convolucionales.

### Resultado del preprocesamiento

Como resultado del proceso de preprocesamiento se obtuvo un conjunto de datos limpio, balanceado y enriquecido con información geométrica y cromática relevante para las tareas de clasificación de calidad y tamaño. La combinación de detección automática, filtrado de recortes inválidos, extracción de características y técnicas de aumento de datos permitió construir una base sólida para el entrenamiento y evaluación de los modelos.

El dataset final quedó compuesto por **36,848 muestras**, cada una asociada a una imagen procesada de tamaño fijo (224×224 píxeles) y a un conjunto de variables descriptivas. En total se almacenaron **24 atributos**, incluyendo información de ubicación, dimensiones, características geométricas, estadísticas de color y etiquetas de clasificación.

Para los modelos clásicos (SVM y XGBoost) se seleccionaron nueve características principales: area_px, aspect_ratio, coverage_ratio, hue_mean, saturation_mean, value_mean, hue_std, saturation_std y value_std. Estas variables combinan información morfológica y cromática, permitiendo representar tanto el tamaño como el estado superficial de las frutas.

La distribución final de las clases de calidad fue relativamente equilibrada, con **14,966 muestras etiquetadas como good**, **11,346 como bad** y **10,536 como regular**. De forma similar, las categorías de tamaño quedaron distribuidas en **14,511 muestras grandes**, **12,053 medianas** y **10,284 pequeñas**, lo que garantiza una representación adecuada de todas las clases durante el entrenamiento.

En conclusión, el proceso de preprocesamiento permitió transformar las imágenes originales en un dataset estructurado, balanceado y adecuado para tareas de aprendizaje automático y aprendizaje profundo, proporcionando la base necesaria para el entrenamiento de los modelos SVM, XGBoost y CNN empleados en este proyecto.
